# Lesson 4.2 — Few-Shot Prompting to Improve Consistency & Quality

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain when few-shot examples outperform detailed instructions alone
- Create 2-4 examples for ambiguous cases showing the reasoning for a choice
- Use examples to demonstrate output format and reduce hallucination in extraction
- Show examples that distinguish acceptable patterns from genuine issues

**You will produce:** A few-shot block for ambiguous tool selection plus a format-demonstrating extraction example.

## Concepts

**Few-shot examples are the most effective technique** for consistent, actionable output when detailed instructions alone produce inconsistent results. They:

- demonstrate **ambiguous-case handling** (tool selection for ambiguous requests, branch-level coverage gaps),
- let the model **generalize judgment** to novel patterns (not just memorize cases),
- **reduce hallucination** in extraction (informal measurements, varied document structures),
- pin down a **specific output format** (location, issue, severity, suggested fix).

Craft **2-4 targeted examples** for ambiguous scenarios that show the **reasoning** for why one action was chosen over plausible alternatives, and include examples distinguishing acceptable code patterns from genuine issues to cut false positives while enabling generalization.

> **Reference:** [Anthropic - multishot prompting](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/multishot-prompting)

In [ ]:
# !pip install anthropic
import os, json

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {str(prompt)[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Few-shot for ambiguous tool selection

Each example shows the query, the chosen tool, and WHY - so the model generalizes.

In [ ]:
few_shot = [
    {"q": "check my order #12345", "tool": "lookup_order",
     "why": "has an order number -> order lookup"},
    {"q": "verify me by email", "tool": "get_customer",
     "why": "identity verification -> customer lookup"},
    {"q": "I ordered something last week, where is it?", "tool": "get_customer",
     "why": "no order number yet -> verify customer first"},
]
def shots_block(shots):
    return "\n".join(f"Q: {s['q']}\n-> {s['tool']} ({s['why']})" for s in shots)

print(shots_block(few_shot))

## Code-Along: Step 2 — Examples that demonstrate output format

Showing the exact shape (location/issue/severity/fix) yields consistent findings.

In [ ]:
format_example = {
    "location": "orders.py:42",
    "issue": "None deref when customer has no orders",
    "severity": "high",
    "suggested_fix": "guard with `if orders:` before indexing",
}
import json
print("Format every finding like:")
print(json.dumps(format_example, indent=2))

## Code-Along: Step 3 — Acceptable pattern vs genuine issue

A contrast pair reduces false positives while letting the model generalize.

In [ ]:
contrast = [
    {"code": "x = data.get('k', 0)", "verdict": "acceptable (safe default)"},
    {"code": "x = data['k']", "verdict": "issue (KeyError risk on optional field)"},
]
for c in contrast:
    print(f"{c['code']:<28} -> {c['verdict']}")

## Exercise

Build a 3-example few-shot block for extracting a measurement field from documents with **varied formats** (e.g. '5kg', 'five kilograms', '5,000 g'), each normalizing to grams. Then add one example that shows returning null when the measurement is absent (ties to Lesson 4.3 nullable fields).

In [ ]:
# TODO: few-shot examples mapping varied measurements -> normalized grams
# TODO: include an 'absent -> null' example to prevent fabrication

## Key Takeaways

- Few-shot examples are the top technique for consistent, actionable output.
- Use 2-4 examples for ambiguous cases, showing the reasoning for the choice.
- Examples demonstrate exact output format and reduce extraction hallucination.
- Contrast acceptable patterns vs genuine issues to cut false positives and generalize.

---

**Next:** [Lesson 4.3 — Structured Output via tool_use & JSON Schema](./lesson-4.3-structured-output-tool-use.ipynb)